In [8]:
import torch

In [9]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU non détecté")


True
Tesla T4


In [10]:
!pip install transformers datasets seaborn -q


In [11]:
import json
import numpy as np
import torch
from pathlib import Path
from google.colab import files
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset, DatasetDict
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict

print(f"GPU disponible : {torch.cuda.is_available()}")
print(f"Device : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

GPU disponible : True
Device : Tesla T4


In [12]:
print("Upload les 4 fichiers : train.json, val.json, test.json, label2id.json")
uploaded = files.upload()

Upload les 4 fichiers : train.json, val.json, test.json, label2id.json


KeyboardInterrupt: 

In [ ]:
with open("label2id.json") as f:
    LABEL2ID = json.load(f)
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

print(f"Labels : {LABEL2ID}")

def charger_split(nom_fichier):
    with open(nom_fichier, encoding="utf-8") as f:
        data = json.load(f)
    return Dataset.from_list([
        {
            "input_ids": d["input_ids"],
            "ner_tags":  d["ner_tags"],
            "id":        d["id"],
            "annee":     d["annee"]
        }
        for d in data
    ])

dataset = DatasetDict({
    "train": charger_split("train.json"),
    "val":   charger_split("val.json"),
    "test":  charger_split("test.json"),
})

print(f"\nTrain : {len(dataset['train'])} docs")
print(f"Val   : {len(dataset['val'])} docs")
print(f"Test  : {len(dataset['test'])} docs")


In [ ]:
def extraire_spans(ner_tags):
    spans = set()
    i = 0
    while i < len(ner_tags):
        tag = ner_tags[i] if isinstance(ner_tags[i], str) else ID2LABEL[ner_tags[i]]
        if tag.startswith("B-"):
            entite = tag[2:]
            debut = i
            i += 1
            while i < len(ner_tags):
                t = ner_tags[i] if isinstance(ner_tags[i], str) else ID2LABEL[ner_tags[i]]
                if t == f"I-{entite}":
                    i += 1
                else:
                    break
            spans.add((entite, debut, i))
        else:
            i += 1
    return spans

def compute_metrics_spans(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=2)

    stats = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
    stats_global = {"tp": 0, "fp": 0, "fn": 0}

    for pred_seq, label_seq in zip(predictions, labels):
        # Ignorer les tokens masqués (-100)
        pred_clean  = [ID2LABEL[p] for p, l in zip(pred_seq, label_seq) if l != -100]
        label_clean = [ID2LABEL[l] for l in label_seq if l != -100]

        spans_pred = extraire_spans(pred_clean)
        spans_gold = extraire_spans(label_clean)

        for span in spans_pred:
            if span in spans_gold:
                stats[span[0]]["tp"] += 1
                stats_global["tp"] += 1
            else:
                stats[span[0]]["fp"] += 1
                stats_global["fp"] += 1

        for span in spans_gold:
            if span not in spans_pred:
                stats[span[0]]["fn"] += 1
                stats_global["fn"] += 1

    def prf(tp, fp, fn):
        p  = tp / (tp + fp) if (tp + fp) > 0 else 0
        r  = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2*p*r / (p+r)  if (p + r)  > 0 else 0
        return p, r, f1

    resultats = {}
    for entite in ["PER", "ORG", "LOC", "MISC"]:
        s = stats[entite]
        p, r, f1 = prf(s["tp"], s["fp"], s["fn"])
        resultats[f"f1_{entite}"] = round(f1 * 100, 2)

    p_g, r_g, f1_g = prf(
        stats_global["tp"], stats_global["fp"], stats_global["fn"]
    )
    resultats["f1_global"]        = round(f1_g * 100, 2)
    resultats["precision_global"] = round(p_g  * 100, 2)
    resultats["recall_global"]    = round(r_g  * 100, 2)
    return resultats


In [ ]:
def preprocess_for_trainer(examples):
    """
    Le Trainer HuggingFace attend :
    - input_ids  : liste d'entiers
    - labels     : liste d'entiers (-100 pour les tokens spéciaux)
    - attention_mask : liste de 0/1
    """
    MAX_LENGTH = 512
    batch_input_ids      = []
    batch_attention_mask = []
    batch_labels         = []

    for input_ids, ner_tags in zip(examples["input_ids"], examples["ner_tags"]):
        # Tronquer si nécessaire
        input_ids = input_ids[:MAX_LENGTH]
        ner_tags  = ner_tags[:MAX_LENGTH]

        # Padding jusqu'à MAX_LENGTH
        pad_len = MAX_LENGTH - len(input_ids)
        attention_mask = [1] * len(input_ids) + [0] * pad_len
        input_ids      = input_ids + [1] * pad_len   # <pad> = 1 pour CamemBERT
        labels         = ner_tags  + [-100] * pad_len

        batch_input_ids.append(input_ids)
        batch_attention_mask.append(attention_mask)
        batch_labels.append(labels)

    return {
        "input_ids":       batch_input_ids,
        "attention_mask":  batch_attention_mask,
        "labels":          batch_labels,
    }

dataset_processed = dataset.map(
    preprocess_for_trainer,
    batched=True,
    remove_columns=["ner_tags", "id", "annee"]
)
dataset_processed.set_format("torch")
print("Dataset préparé pour le Trainer.")


In [ ]:
MODEL_1 = "camembert-base"
print(f"\nChargement de {MODEL_1}...")

model_1 = AutoModelForTokenClassification.from_pretrained(
    MODEL_1,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

In [ ]:
args_1 = TrainingArguments(
    output_dir              = "./camembert_base_ner",
    num_train_epochs        = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate           = 2e-5,
    weight_decay            = 0.01,
    warmup_ratio            = 0.1,
    evaluation_strategy     = "epoch",
    save_strategy           = "epoch",
    load_best_model_at_end  = True,
    metric_for_best_model   = "f1_global",
    greater_is_better       = True,
    logging_steps           = 100,
    fp16                    = True,   # accélération GPU
    report_to               = "none",
    save_total_limit        = 2,
)

trainer_1 = Trainer(
    model           = model_1,
    args            = args_1,
    train_dataset   = dataset_processed["train"],
    eval_dataset    = dataset_processed["val"],
    compute_metrics = compute_metrics_spans,
)

print("Démarrage de l'entraînement CamemBERT-base...")
trainer_1.train()


In [ ]:
print("\n=== ÉVALUATION FINALE CamemBERT-base sur TEST ===")
results_1 = trainer_1.evaluate(dataset_processed["test"])

print(f"\n{'Métrique':<25} {'Score':>10}")
print("-" * 38)
for k, v in results_1.items():
    if "f1" in k or "precision" in k or "recall" in k:
        print(f"{k:<25} {v:>9.2f}%")

# Sauvegarder
with open("results_camembert_base.json", "w") as f:
    json.dump(results_1, f, indent=2)
print("\nRésultats sauvegardés → results_camembert_base.json")

In [ ]:
MODEL_2 = "Jean-Baptiste/camembert-ner"
print(f"\nChargement de {MODEL_2}...")

model_2 = AutoModelForTokenClassification.from_pretrained(
    MODEL_2,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

In [ ]:
args_2 = TrainingArguments(
    output_dir              = "./camembert_ner_finetuned",
    num_train_epochs        = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate           = 1e-5,   # plus faible car déjà spécialisé
    weight_decay            = 0.01,
    warmup_ratio            = 0.1,
    evaluation_strategy     = "epoch",
    save_strategy           = "epoch",
    load_best_model_at_end  = True,
    metric_for_best_model   = "f1_global",
    greater_is_better       = True,
    logging_steps           = 100,
    fp16                    = True,
    report_to               = "none",
    save_total_limit        = 2,
)

trainer_2 = Trainer(
    model           = model_2,
    args            = args_2,
    train_dataset   = dataset_processed["train"],
    eval_dataset    = dataset_processed["val"],
    compute_metrics = compute_metrics_spans,
)

print("Démarrage de l'entraînement CamemBERT-NER...")
trainer_2.train()

In [ ]:
print("\n=== ÉVALUATION FINALE CamemBERT-NER sur TEST ===")
results_2 = trainer_2.evaluate(dataset_processed["test"])

print(f"\n{'Métrique':<25} {'Score':>10}")
print("-" * 38)
for k, v in results_2.items():
    if "f1" in k or "precision" in k or "recall" in k:
        print(f"{k:<25} {v:>9.2f}%")

with open("results_camembert_ner.json", "w") as f:
    json.dump(results_2, f, indent=2)
print("\nRésultats sauvegardés → results_camembert_ner.json")


In [ ]:
baseline = {
    "eval_f1_PER": 11.68, "eval_f1_ORG": 8.52,
    "eval_f1_LOC": 5.63,  "eval_f1_MISC": 0.15,
    "eval_f1_global": 6.44,
    "eval_precision_global": 3.49, "eval_recall_global": 41.3
}

print("\n" + "=" * 75)
print("TABLEAU COMPARATIF FINAL — 3 MODÈLES")
print("=" * 75)
print(f"{'Modèle':<30} {'F1-PER':>8} {'F1-ORG':>8} {'F1-LOC':>8} {'F1-MISC':>9} {'F1-Global':>11}")
print("-" * 75)

modeles = [
    ("spaCy baseline",       baseline),
    ("CamemBERT-base",       results_1),
    ("CamemBERT-NER",        results_2),
]

for nom, res in modeles:
    f1_per  = res.get("eval_f1_PER",    res.get("f1_PER",  0))
    f1_org  = res.get("eval_f1_ORG",    res.get("f1_ORG",  0))
    f1_loc  = res.get("eval_f1_LOC",    res.get("f1_LOC",  0))
    f1_misc = res.get("eval_f1_MISC",   res.get("f1_MISC", 0))
    f1_glob = res.get("eval_f1_global", res.get("f1_global", 0))
    print(f"{nom:<30} {f1_per:>7.1f}% {f1_org:>7.1f}% {f1_loc:>7.1f}% {f1_misc:>8.1f}% {f1_glob:>10.1f}%")

print("=" * 75)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Graphique 1 — F1 par entité et par modèle
entites  = ["PER", "ORG", "LOC", "MISC"]
noms     = ["spaCy\nbaseline", "CamemBERT\nbase", "CamemBERT\nNER"]
couleurs = ["#ef4444", "#4a9eed", "#22c55e"]

x = np.arange(len(entites))
width = 0.25

for i, (nom, res, couleur) in enumerate(zip(noms, [baseline, results_1, results_2], couleurs)):
    scores = [
        res.get("eval_f1_PER",  res.get("f1_PER",  0)),
        res.get("eval_f1_ORG",  res.get("f1_ORG",  0)),
        res.get("eval_f1_LOC",  res.get("f1_LOC",  0)),
        res.get("eval_f1_MISC", res.get("f1_MISC", 0)),
    ]
    axes[0].bar(x + i*width, scores, width, label=nom.replace("\n"," "),
                color=couleur, alpha=0.85)

axes[0].set_xlabel("Type d'entité")
axes[0].set_ylabel("F1 Score (%)")
axes[0].set_title("F1 par entité — Comparaison des modèles")
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(entites)
axes[0].legend()
axes[0].set_ylim(0, 100)
axes[0].grid(axis="y", alpha=0.3)

# Graphique 2 — F1 global
f1_globaux = [
    baseline.get("eval_f1_global", 6.44),
    results_1.get("eval_f1_global", 0),
    results_2.get("eval_f1_global", 0),
]
bars = axes[1].bar(
    ["spaCy\nbaseline", "CamemBERT\nbase", "CamemBERT\nNER"],
    f1_globaux,
    color=["#ef4444", "#4a9eed", "#22c55e"],
    alpha=0.85, width=0.5
)
for bar, val in zip(bars, f1_globaux):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f"{val:.1f}%", ha="center", fontsize=13, fontweight="bold")

axes[1].set_ylabel("F1 Global (%)")
axes[1].set_title("F1 Global — Comparaison des modèles")
axes[1].set_ylim(0, 100)
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("NER Archelec — Résultats comparatifs", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("comparaison_modeles.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graphique sauvegardé → comparaison_modeles.png")

In [ ]:
files.download("results_camembert_base.json")
files.download("results_camembert_ner.json")
files.download("comparaison_modeles.png")
print("\nFichiers téléchargés. Copie-les dans projet_archelec/models/results/")